# IT7075 — Compare Self-Hosted LLMs on Your NAIRR VM

This is the **"a GPU you keep"** path (Way 3). Your NAIRR VM is a *remote*
machine — the models run on its A100 slice, not on your laptop — but unlike a Colab
session, the VM (and everything you install) **persists all term**. Here you talk to the
models you self-hosted with **Ollama** and compare two of them on the same prompt.

Run it in **VS Code (Remote-SSH)** with the kernel **on the VM**, so it reaches Ollama at
`localhost:11434`.

**Before you start** (see the Cloud VM lab + the Private LLM lab):
- Provision and connect to your VM (the Tools & Environment module, Lab 2A).
- Install Ollama on the VM and pull at least two models, e.g. `llama3.1:8b` and `qwen2.5:14b`.

> Contrast with `host_LLM_GPU.ipynb`, which self-hosts via Hugging Face `transformers`
> (that notebook's GPU code also runs here on the VM). Ollama is the quickest way to stand
> up a persistent, browser-accessible model server (Open WebUI) that's still there tomorrow.

In [1]:
import requests, time

OLLAMA = "http://localhost:11434"

def list_models():
    """Show the models available locally."""
    r = requests.get(f"{OLLAMA}/api/tags")
    return [m["name"] for m in r.json().get("models", [])]

list_models()

['qwen2.5:14b', 'llama3.2:1b', 'llama3.1:8b']

In [2]:
def ask(model, prompt, temperature=0.7):
    """Send a prompt to one model and return (text, seconds)."""
    t0 = time.time()
    r = requests.post(f"{OLLAMA}/api/generate", json={
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature},
    })
    elapsed = time.time() - t0
    return r.json()["response"], elapsed

## Pick two models to compare

Change these to any two names printed by `list_models()` above.

In [3]:
MODEL_A = "llama3.1:8b"
MODEL_B = "qwen2.5:14b"

PROMPT = "Explain what a reverse shell is, and one way defenders detect it. Two sentences."

for model in (MODEL_A, MODEL_B):
    text, secs = ask(model, PROMPT)
    print(f"\n===== {model}  ({secs:.1f}s) =====")
    print(text.strip())


===== llama3.1:8b  (24.7s) =====
A reverse shell is a type of remote access tool (RAT) that allows an attacker to establish a command-line interface (CLI) on their own system by executing a malicious payload on the compromised host, which connects back to the attacker's control server over a network connection. Defenders can detect reverse shells through behavioral analysis, such as monitoring for suspicious outbound connections or network traffic patterns indicative of a reverse shell communication protocol like DNS tunneling or TCP port 443 usage.



===== qwen2.5:14b  (33.2s) =====
A reverse shell is a type of hacking technique where an attacker establishes control over a target computer by executing a command or script on the victim's machine that connects back to the attacker's machine, allowing them to execute commands and access resources on the compromised system; network intrusion detection systems can detect reverse shells by monitoring for unusual outbound connections from internal networks to external IPs on uncommon ports.


## Your turn

1. Change `PROMPT` to a security question of your own and re-run the comparison.
2. Set `temperature=0.0` in `ask(...)` and then `temperature=1.0`. How does the
   output change? Which is better for a factual security question?
3. Add a third model to the loop (pull one first with `ollama pull`).
4. Note which model is faster and which gives the better answer — they are often
   not the same model. Record your comparison for the lab submission.